# Project health analysis (yt-dlp)
Needs `issues_prs.csv` and `commits.csv` (from `HealthDataExport.cs`) in the same folder.

## Step 1: Approach and data

In [90]:
import pandas as pd

# Our year (the same twelve months as in Section 6) and the year before, for comparison
START, END = "2025-09-01", "2026-08-31"
PREV_START, PREV_END = "2024-09-01", "2025-08-31"

# keep_default_na=False: empty cells stay empty text instead of becoming NaN
issues_prs = pd.read_csv("issues_prs.csv", keep_default_na=False)
commits = pd.read_csv("commits.csv", keep_default_na=False)

def in_period(dates, start, end):
    # The dates are text like "2025-09-16 09:20:16", so we compare only the day part
    day = dates.str[:10]
    return (day >= start) & (day <= end)

print("Issues and pull requests:", len(issues_prs),
      "| from", issues_prs["CreatedAt"].min()[:10], "to", issues_prs["CreatedAt"].max()[:10])
print("  of which issues:", (issues_prs["Type"] == "Issue").sum(), "| pull requests:", (issues_prs["Type"] == "PR").sum())
print("Commits:", len(commits), "| oldest from", commits["AuthorDate"].min()[:10])

# Commits from before 2021 come from youtube-dl, the project yt-dlp was forked from. We leave them out.
commits = commits[commits["AuthorDate"].str[:10] >= "2021-01-01"]
print("Commits since yt-dlp started (January 2021):", len(commits))

year = issues_prs[in_period(issues_prs["CreatedAt"], START, END)]
print()
print("In our year:", (year["Type"] == "Issue").sum(), "issues and", (year["Type"] == "PR").sum(), "pull requests")
print("Commits in our year:", in_period(commits["AuthorDate"], START, END).sum())

Issues and pull requests: 17297 | from 2021-01-06 to 2026-10-13
  of which issues: 12143 | pull requests: 5154
Commits: 24011 | oldest from 2008-07-21
Commits since yt-dlp started (January 2021): 5620

In our year: 2210 issues and 1093 pull requests
Commits in our year: 541


## Step 2: Filtering bots and multiple accounts
### Bots

In [91]:
import re

# Measure 1a: the label GitHub itself gives to an account ("User" or "Bot")
labelled_bot = issues_prs[issues_prs["AuthorType"] == "Bot"]
print("Issues/PRs whose author GitHub labels as 'Bot':")
print(labelled_bot[["Number", "Type", "Author", "Title"]].to_string(index=False))

# Measure 1b: accounts with "bot" in their name (issue/PR authors and commit authors)
names = set(issues_prs["Author"]) | set(commits["AuthorLogin"])
bot_names = sorted(n for n in names if "bot" in n.lower())
print()
print(len(bot_names), "accounts have 'bot' in their name:")
print(bot_names)

Issues/PRs whose author GitHub labels as 'Bot':
 Number Type  Author                                                              Title
  15074   PR Copilot [WIP] Add example script to download playlist as FLAC with artwork

20 accounts have 'bot' in their name:
['Ana-Robot', 'BotTG1630', 'BotTutorials', 'Botato300', 'DKBOTZPROJECT', 'Lee559-bot', 'Mintish-Bot', 'MrBotDeveloper', 'RabotaDime', 'RobotJNate', 'Robotic-Brain', 'SubhutiBottleneck', 'arnot-bot', 'bothie', 'botsmaster', 'charleswesleybotchwey', 'dlp-bot', 'github-actions[bot]', 'keboto', 'mango-bots']


In [92]:
# Measure 2: templated titles.
# A bot uses the same sentence again and again, only with other numbers.
# So we replace every number by # and count how many different title "shapes" an account has.
def title_shape(title):
    title = re.sub(r"\(#\d+\)$", "", title.strip())   # remove the PR number at the end of commit titles
    title = re.sub(r"\d+", "#", title)                # every number becomes #
    return title.strip()

# One row per title, with the account that wrote it (commits, issues and pull requests)
titles = pd.concat([
    pd.DataFrame({"Account": commits["AuthorLogin"], "Title": commits["Title"]}),
    pd.DataFrame({"Account": issues_prs["Author"], "Title": issues_prs["Title"]}),
])
titles = titles[(titles["Account"] != "") & (titles["Account"] != "ghost")]
titles["Shape"] = titles["Title"].apply(title_shape)

per_account = titles.groupby("Account").agg(Titles=("Title", "size"), Shapes=("Shape", "nunique"))
per_account["Share"] = (per_account["Shapes"] / per_account["Titles"]).round(2)

# With only a few titles the share says little (one title is always 100%),
# so we run this measure on every account with at least 10 titles
enough = per_account[per_account["Titles"] >= 10].sort_values("Share")
print(len(enough), "accounts have at least 10 titles. The ones with the most repeated titles:")
print(enough.head(5).to_string())

177 accounts have at least 10 titles. The ones with the most repeated titles:
                     Titles  Shapes  Share
Account                                   
github-actions[bot]      60       1   0.02
dlp-bot                  33       4   0.12
bashonly               1722     963   0.56
seproDev                433     268   0.62
jshumphrey               12       8   0.67


In [93]:
# The accounts from measure 1b, checked with measure 2
print("Accounts with 'bot' in their name:")
print(per_account.loc[[n for n in bot_names if n in per_account.index]].sort_values("Titles", ascending=False).to_string())

Accounts with 'bot' in their name:
                       Titles  Shapes  Share
Account                                     
github-actions[bot]        60       1   0.02
dlp-bot                    33       4   0.12
Botato300                   6       6   1.00
Ana-Robot                   3       3   1.00
SubhutiBottleneck           3       3   1.00
keboto                      2       2   1.00
MrBotDeveloper              2       2   1.00
charleswesleybotchwey       1       1   1.00
botsmaster                  1       1   1.00
bothie                      1       1   1.00
arnot-bot                   1       1   1.00
Robotic-Brain               1       1   1.00
BotTG1630                   1       1   1.00
RobotJNate                  1       1   1.00
RabotaDime                  1       1   1.00
Mintish-Bot                 1       1   1.00
Lee559-bot                  1       1   1.00
DKBOTZPROJECT               1       1   1.00
BotTutorials                1       1   1.00
mango-bots          

In [94]:
# Rows for the table in the report:
# the two templated accounts, the first account after the gap, and the account GitHub labels as a bot
gap = list(enough.index[:3])
github_bot = list(labelled_bot["Author"].unique())

for account in gap + github_bot:
    label = set(issues_prs["AuthorType"][issues_prs["Author"] == account]) or "-"
    opened_issue = ((issues_prs["Author"] == account) & (issues_prs["Type"] == "Issue")).any()
    row = per_account.loc[account]
    print(f"{account:20} label: {str(label):9} titles: {int(row['Titles']):5} shapes: {int(row['Shapes']):4} opened an issue: {opened_issue}")
    print("    most common shape:", titles[titles["Account"] == account]["Shape"].value_counts().index[0])

github-actions[bot]  label: -         titles:    60 shapes:    1 opened an issue: False
    most common shape: Release #.#.#
dlp-bot              label: {'User'}  titles:    33 shapes:    4 opened an issue: False
    most common shape: [build] Update # dependencies
bashonly             label: {'User'}  titles:  1722 shapes:  963 opened an issue: True
    most common shape: [cleanup] Misc
Copilot              label: {'Bot'}   titles:     1 shapes:    1 opened an issue: False
    most common shape: [WIP] Add example script to download playlist as FLAC with artwork


In [95]:
# Decision: a bot if GitHub labels it "Bot", or if its name says bot AND its titles are templated.
# dlp-bot also uses the email bot@yt-dlp.org:
print("dlp-bot emails:", set(commits["AuthorEmail"][commits["AuthorLogin"] == "dlp-bot"]))

BOTS = {"github-actions[bot]", "dlp-bot", "copilot"}   # lower case, we compare names in lower case

def is_bot(name):
    return name.lower() in BOTS

commits_year = commits[in_period(commits["AuthorDate"], START, END)]
prs_year = issues_prs[in_period(issues_prs["CreatedAt"], START, END) & (issues_prs["Type"] == "PR")]
issues_year = issues_prs[in_period(issues_prs["CreatedAt"], START, END) & (issues_prs["Type"] == "Issue")]

print()
print("Removed in our year:")
for bot in sorted(BOTS):
    print(f"  {bot:20} commits: {commits_year['AuthorLogin'].str.lower().eq(bot).sum():3}  "
          f"PRs: {prs_year['Author'].str.lower().eq(bot).sum():3}  issues: {issues_year['Author'].str.lower().eq(bot).sum()}")
print("Commits in our year:", len(commits_year), "->", (~commits_year["AuthorLogin"].apply(is_bot)).sum())
print("PRs in our year:", len(prs_year), "->", (~prs_year["Author"].apply(is_bot)).sum())

dlp-bot emails: {'bot@yt-dlp.org'}

Removed in our year:
  copilot              commits:   0  PRs:   1  issues: 0
  dlp-bot              commits:  15  PRs:  16  issues: 0
  github-actions[bot]  commits:  17  PRs:   0  issues: 0
Commits in our year: 541 -> 509
PRs in our year: 1093 -> 1076


### Multiple accounts of the same person

In [96]:
# Only our year, without bots
cy = commits_year[~commits_year["AuthorLogin"].apply(is_bot)]

# Measure 1: git names that differ from the GitHub account they are linked to
linked = cy[cy["AuthorLogin"] != ""]
different = linked[linked["AuthorName"].str.lower().str.replace(" ", "") != linked["AuthorLogin"].str.lower()]
print("Git name -> GitHub account (number of commits), only where they differ:")
print(different.groupby(["AuthorName", "AuthorLogin"]).size().sort_values(ascending=False).head(8).to_string())

Git name -> GitHub account (number of commits), only where they differ:
AuthorName     AuthorLogin
sepro          seproDev       57
Simon Sawicki  Grub4K         16
0x∅            0xvd           12
Subrat Lima    subrat-lima     3
Zer0 Spectrum  0xvd            3
Robin          robin-mu        2
N/Ame          grqz            2
Deltadroid     syphyr          2


In [97]:
# Measure 2a: commits without a linked account -> who does "Authored by" say?
print("Commits without a linked GitHub account:")
print(cy[cy["AuthorLogin"] == ""][["Sha", "AuthorName", "AuthoredBy", "Title"]].to_string(index=False))

# Measure 2b: commits where "Authored by" names someone else than the linked account (one author only)
single = cy[(cy["AuthoredBy"] != "") & ~cy["AuthoredBy"].str.contains(",") & (cy["AuthorLogin"] != "")]
mismatch = single[single["AuthoredBy"].str.lower() != single["AuthorLogin"].str.lower()]
print()
print("'Authored by' differs from the linked account:")
print(mismatch.groupby(["AuthorLogin", "AuthoredBy"]).size().to_string())

# The "Zer0spectrum" commits: GitHub links them to 0xvd through their email.
# Which other commits use the same email, and whose name is in their "Authored by" line?
zer0 = cy[cy["AuthoredBy"].str.lower() == "zer0spectrum"]
print()
print(zer0[["Sha", "AuthorName", "AuthorLogin", "AuthoredBy", "Title"]].to_string(index=False))
same_email = cy[cy["AuthorEmail"].isin(zer0["AuthorEmail"])]
print()
print("All commits in our year with the same email (git name, first name after 'Authored by'):")
print(same_email.groupby([same_email["AuthorName"], same_email["AuthoredBy"].str.split(",").str[0]]).size().to_string())

Commits without a linked GitHub account:
    Sha               AuthorName                         AuthoredBy                                                          Title
e311860                      0x∅                               0xvd                       [ie/thechosen] Rework extractor (#16021)
133cb95                      0x∅                               0xvd                           [ie/xhamster] Fix extractor (#15831)
715af0c WhatAmISupposedToPutHere WhatAmISupposedToPutHere, bashonly [ie/youtube] Determine wait time from player response (#14646)

'Authored by' differs from the linked account:
AuthorLogin  AuthoredBy  
0xvd         Zer0spectrum    3
bashonly     Grub4K          1
seproDev     cdce8p          1

    Sha    AuthorName AuthorLogin   AuthoredBy                                                                 Title
2a777ec Zer0 Spectrum        0xvd Zer0spectrum                             [ie/tubitv:series] Fix extractor (#15018)
4433b3a Zer0 Spectrum        0xvd

In [98]:
# Decision: merge Zer0spectrum and "0x∅" into 0xvd.
# person() gives one name per person: the linked account, otherwise the first name after "Authored by"
SAME_PERSON = {"zer0spectrum": "0xvd", "0x∅": "0xvd"}

def person(name):
    name = name.strip().lower()
    return SAME_PERSON.get(name, name)

def commit_author(row):
    if row["AuthorLogin"] != "":
        return person(row["AuthorLogin"])
    if row["AuthoredBy"] != "":
        return person(row["AuthoredBy"].split(",")[0])
    return person(row["AuthorName"])

print("Commit authors in our year by git name: ", cy["AuthorName"].nunique())
print("Commit authors in our year by email:    ", cy["AuthorEmail"].str.lower().nunique())
print("Commit authors in our year as people:   ", cy.apply(commit_author, axis=1).nunique())

Commit authors in our year by git name:  124
Commit authors in our year by email:     125
Commit authors in our year as people:    121


### One account used by many people

In [99]:
ghost = issues_prs[issues_prs["Author"] == "ghost"]
print("Issues/PRs by ghost:", len(ghost), "| in our year:", in_period(ghost["CreatedAt"], START, END).sum())

web = cy[cy["CommitterName"] == "GitHub"]
print("Commits in our year (no bots) with committer 'GitHub':", len(web), "of", len(cy),
      "| different authors behind it:", web.apply(commit_author, axis=1).nunique())

Issues/PRs by ghost: 289 | in our year: 29
Commits in our year (no bots) with committer 'GitHub': 503 of 509 | different authors behind it: 121


## Step 3: Contributor categories
Contribution = pull request or commit. Reporters = people who only opened issues.

In [100]:
# One row per event: who, when, what
from_issues_prs = pd.DataFrame({
    "Person": issues_prs["Author"].apply(person),
    "Date": issues_prs["CreatedAt"].str[:10],
    "Type": issues_prs["Type"],
})
from_commits = pd.DataFrame({
    "Person": commits.apply(commit_author, axis=1),
    "Date": commits["AuthorDate"].str[:10],
    "Type": "Commit",
})
events = pd.concat([from_issues_prs, from_commits])
events = events[~events["Person"].apply(is_bot) & (events["Person"] != "ghost")]
events["Code"] = events["Type"].isin(["PR", "Commit"])    # True for a contribution (pull request or commit)

year_events = events[(events["Date"] >= START) & (events["Date"] <= END)]
people = set(year_events["Person"])
code_people = set(year_events[year_events["Code"]]["Person"])
print("People active in our year:", len(people))
print("  code contributors (at least one PR or commit):", len(code_people))
print("  reporters (only issues):", len(people - code_people))

People active in our year: 2103
  code contributors (at least one PR or commit): 474
  reporters (only issues): 1629


In [101]:
# Contributions per code contributor in our year, and in how many of the 12 months they contributed
year_code = year_events[year_events["Code"]]
per_person = year_code.groupby("Person").size().sort_values(ascending=False)
active_months = year_code.groupby("Person")["Date"].apply(lambda d: d.str[:7].nunique())
print("Top 10 (contributions, active months):")
print(pd.DataFrame({"Contributions": per_person, "Active months": active_months}).loc[per_person.index[:15]].to_string())

Top 10 (contributions, active months):
                          Contributions  Active months
Person                                                
bashonly                            425             12
doe1080                             116             12
seprodev                            111              6
0xvd                                108             10
grub4k                               27              7
invalidusernameexception             24              9
norepro                               9              3
tcely                                 9              3
grqz                                  7              4
sipherdrakon                          7              3
whatamisupposedtoputhere              6              3
syphyr                                6              4
swayll                                6              3
garret1317                            6              5
gamer191                              6              4


Gap after the top six: threshold for *frequent* = 20.

In [102]:
# The categories, with 20 as the threshold for frequent contributors
frequent = per_person[per_person >= 20]
occasional = per_person[(per_person >= 2) & (per_person < 20)]
one_time = per_person[per_person == 1]
print("Frequent (20 or more):", len(frequent), "| occasional (2-19):", len(occasional),
      "| one-time (1):", len(one_time), f"({len(one_time) / len(per_person):.0%})")

# CHAOSS Contributor Absence Factor: the smallest number of people that together make 50% of all contributions
share = per_person.cumsum() / per_person.sum()
print("Total contributions:", per_person.sum(), "| Contributor Absence Factor:", (share < 0.5).sum() + 1)
print("coletdjnz (core maintainer):", per_person.get("coletdjnz", 0), "contributions")

Frequent (20 or more): 6 | occasional (2-19): 171 | one-time (1): 297 (63%)
Total contributions: 1582 | Contributor Absence Factor: 6
coletdjnz (core maintainer): 6 contributions


In [103]:
# Newcomers: people whose FIRST contribution ever (since January 2021) is in our year
first_contribution = events[events["Code"]].groupby("Person")["Date"].min()
newcomers = [p for p in per_person.index if first_contribution[p] >= START]
print("Newcomers:", len(newcomers), "of", len(per_person), f"({len(newcomers) / len(per_person):.0%})")
print("  of which frequent:", [p for p in newcomers if p in frequent.index],
      "| occasional:", sum(p in occasional.index for p in newcomers),
      "| one-time:", sum(p in one_time.index for p in newcomers))

Newcomers: 395 of 474 (83%)
  of which frequent: ['0xvd'] | occasional: 132 | one-time: 262


In [104]:
# Dormant contributors. How long can someone be quiet and still come back?
# For every person we take the months with a contribution and look at the gaps between them.
def month_number(date):
    return int(date[:4]) * 12 + int(date[5:7])

gaps = []
for p, dates in events[events["Code"]].groupby("Person")["Date"]:
    months = sorted(set(month_number(d) for d in dates))
    gaps += [b - a for a, b in zip(months, months[1:])]
gaps = pd.Series(gaps)
print("Gaps between active months:", len(gaps))
for q in [0.5, 0.75, 0.9, 0.95]:
    print(f"  {q:.0%} of the gaps are at most {int(gaps.quantile(q, interpolation='lower'))} months")

Gaps between active months: 1581
  50% of the gaps are at most 1 months
  75% of the gaps are at most 3 months
  90% of the gaps are at most 9 months
  95% of the gaps are at most 16 months


In [105]:
# 90% of the gaps are 9 months or less, so a whole year without contributing means someone has most likely stopped.
# Dormant: contributed in the year before, but not in our year
before = events[events["Code"] & (events["Date"] >= PREV_START) & (events["Date"] <= PREV_END)]
before_people = set(before["Person"])
dormant = before_people - code_people
print("Code contributors in the year before:", len(before_people))
print("  dormant in our year:", len(dormant), f"({len(dormant) / len(before_people):.0%})")

# Example: the most active person since 2021
top = events[events["Code"]].groupby("Person").size().sort_values(ascending=False)
print()
print("Most contributions since 2021:", top.head(3).to_dict())
print("Last contribution of pukkandan:", events[events["Code"] & (events["Person"] == "pukkandan")]["Date"].max())

Code contributors in the year before: 349
  dormant in our year: 296 (85%)

Most contributions since 2021: {'pukkandan': 1877, 'bashonly': 1708, 'coletdjnz': 438}
Last contribution of pukkandan: 2024-12-23


In [106]:
# Comparison with earlier years (each year runs from September to August)
def season(date):
    year, month = int(date[:4]), int(date[5:7])
    start = year if month >= 9 else year - 1
    return f"{start}/{str(start + 1)[2:]}"

events["Year"] = events["Date"].apply(season)
per_year = pd.DataFrame({
    "Code contributors": events[events["Code"]].groupby("Year")["Person"].nunique(),
    "Issue openers": events[events["Type"] == "Issue"].groupby("Year")["Person"].nunique(),
})
print(per_year.loc["2021/22":"2025/26"].to_string())

         Code contributors  Issue openers
Year                                     
2021/22                328           1655
2022/23                352           1590
2023/24                336           1490
2024/25                349           1651
2025/26                474           1721


## Step 4: Activity metrics
One metric per type of activity, for the year before and our year (bots removed).

In [107]:
def activity(start, end):
    items = issues_prs[in_period(issues_prs["CreatedAt"], start, end) & ~issues_prs["Author"].apply(is_bot)]
    issues = items[items["Type"] == "Issue"]
    prs = items[items["Type"] == "PR"]
    year_commits = commits[in_period(commits["AuthorDate"], start, end) & ~commits["AuthorLogin"].apply(is_bot)]
    # every release creates a "Release ..." commit by github-actions[bot]
    releases = commits[in_period(commits["AuthorDate"], start, end)
                       & (commits["AuthorLogin"] == "github-actions[bot]")
                       & commits["Title"].str.startswith("Release")]
    return {
        "Issues opened per month": round(len(issues) / 12),
        "Issues without any comment (%)": round(100 * (issues["Comments"] == 0).mean()),
        "Pull requests opened per month": round(len(prs) / 12),
        "Commits": len(year_commits),
        "Releases": len(releases),
    }

table = pd.DataFrame({"2024/25": activity(PREV_START, PREV_END), "2025/26": activity(START, END)})
print(table.to_string())

                                2024/25  2025/26
Issues opened per month             183      184
Issues without any comment (%)        8       11
Pull requests opened per month       86       90
Commits                             738      509
Releases                             28       17


## Step 5: Process metrics and sampling
### Pull requests per group (all pull requests of our year)

In [108]:
# Everyone listed in Maintainers.md (core, inactive core, maintainers and triage maintainers)
MAINTAINERS = {"coletdjnz", "bashonly", "grub4k", "pukkandan", "shirt-dev", "ashish0804", "seprodev",
               "doe1080", "gamer191", "garret1317", "pzhlkj6612", "dtrombett", "grqz", "invalidusernameexception"}

def group(author):
    if author in MAINTAINERS:
        return "maintainer"
    if first_contribution[author] >= START:
        return "newcomer"
    return "returning"

prs = issues_prs[in_period(issues_prs["CreatedAt"], START, END) & (issues_prs["Type"] == "PR")].copy()
prs["Person"] = prs["Author"].apply(person)
prs = prs[~prs["Person"].apply(is_bot) & (prs["Person"] != "ghost")]
# 0xvd is a newcomer by date but became a frequent contributor (Step 3), so we leave him out
prs = prs[prs["Person"] != "0xvd"]
prs["Group"] = prs["Person"].apply(group)

created = pd.to_datetime(prs["CreatedAt"])
merged_at = pd.to_datetime(prs["MergedAt"].replace("", None))
closed_at = pd.to_datetime(prs["ClosedAt"].replace("", None))
prs["Outcome"] = "open"
prs.loc[prs["State"] == "closed", "Outcome"] = "closed without merge"
prs.loc[prs["MergedAt"] != "", "Outcome"] = "merged"
prs["Days to merge"] = (merged_at - created).dt.total_seconds() / 86400
prs["Days to close"] = (closed_at - created).dt.total_seconds() / 86400

rows = {}
for g, part in prs.groupby("Group"):
    rows[g] = {
        "PRs": len(part),
        "People": int(part["Person"].nunique()),
        "Merged %": int(round(100 * (part["Outcome"] == "merged").mean())),
        "Closed without merge %": int(round(100 * (part["Outcome"] == "closed without merge").mean())),
        "Still open %": int(round(100 * (part["Outcome"] == "open").mean())),
        "Median days to merge": float(round(part["Days to merge"].median(), 2)),
        "Median days to close without merge": float(round(part.loc[part["Outcome"] == "closed without merge", "Days to close"].median(), 2)),
    }
print(pd.DataFrame.from_dict(rows, orient="index").loc[["newcomer", "returning", "maintainer"]].to_string())

            PRs  People  Merged %  Closed without merge %  Still open %  Median days to merge  Median days to close without merge
newcomer    514     393        17                      50            33                  7.64                                0.06
returning   100      62        26                      19            55                  3.63                                1.73
maintainer  375      10        89                       1            10                  0.34                               75.08


### Stratified random sample: 50 pull requests per group

In [109]:
import random

# The same seed gives the same "random" sample every time
random.seed(42)
sample_rows = []
for g in ["newcomer", "returning", "maintainer"]:
    numbers = list(prs[prs["Group"] == g]["Number"])
    for number in sorted(random.sample(numbers, 50)):
        pr = prs[prs["Number"] == number].iloc[0]
        sample_rows.append({"Number": number, "Group": g, "Author": pr["Author"], "CreatedAt": pr["CreatedAt"]})

sample = pd.DataFrame(sample_rows)
sample.to_csv("sample_prs.csv", index=False)   # input for FirstResponseExport.cs
print(sample["Group"].value_counts().to_string())

Group
newcomer      50
returning     50
maintainer    50


### Time to first response
`first_responses.csv` is made by `FirstResponseExport.cs` from `sample_prs.csv`.

In [110]:
responses = pd.read_csv("first_responses.csv", keep_default_na=False)
responses["Hours"] = (pd.to_datetime(responses["FirstResponseAt"].replace("", None))
                      - pd.to_datetime(responses["CreatedAt"])).dt.total_seconds() / 3600

rows = {}
for g, part in responses.groupby("Group"):
    rows[g] = {
        "Got a response": int((part["FirstResponseAt"] != "").sum()),
        "Within 1 day": int((part["Hours"] <= 24).sum()),
        "Within 1 week": int((part["Hours"] <= 24 * 7).sum()),
        "Median hours (if answered)": float(round(part["Hours"].median(), 1)),
        "Reviews missing": int((part["ReviewsMissing"] == "yes").sum()),
    }
print(pd.DataFrame.from_dict(rows, orient="index").loc[["newcomer", "returning", "maintainer"]].to_string())

            Got a response  Within 1 day  Within 1 week  Median hours (if answered)  Reviews missing
newcomer                38            26             32                         2.8               14
returning               29            11             19                        57.3                2
maintainer              38            28             35                         6.0                0


In [111]:
# What happened to the pull requests without any response?
outcome = prs.set_index("Number")["Outcome"]
no_response = responses[responses["FirstResponseAt"] == ""]
print(no_response.groupby(["Group", no_response["Number"].map(outcome)]).size().to_string())

# Who responds first?
answered = responses[responses["FirstResponseAt"] != ""]
print()
print("First responders:", answered["FirstResponder"].value_counts().head(3).to_dict(), "of", len(answered))

Group       Number              
maintainer  merged                   9
            open                     3
newcomer    closed without merge     7
            open                     5
returning   closed without merge     4
            open                    17

First responders: {'bashonly': 42, 'seproDev': 15, 'Grub4K': 11} of 105


## Step 6: Qualitative analysis
### Targeted sample: the pull requests of the random sample that show the two problems

In [112]:
status = prs.set_index("Number")
reading = responses.copy()
reading["Outcome"] = reading["Number"].map(status["Outcome"])
reading["Title"] = reading["Number"].map(status["Title"])

newcomer_closed = reading[(reading["Group"] == "newcomer") & (reading["Outcome"] == "closed without merge")]
returning_ignored = reading[(reading["Group"] == "returning") & (reading["Outcome"] == "open")
                            & (reading["FirstResponseAt"] == "")]
print("Newcomer PRs closed without merge:", len(newcomer_closed))
print("Returning PRs still open without any response:", len(returning_ignored))

reading_list = pd.concat([newcomer_closed.assign(Why="newcomer-closed"),
                          returning_ignored.assign(Why="returning-open-no-response")])
reading_list["Link"] = "https://github.com/yt-dlp/yt-dlp/pull/" + reading_list["Number"].astype(str)
reading_list[["Number", "Why", "Author", "Title", "Link"]].to_csv("reading_list.csv", index=False)  # input for ReadingExport.cs

Newcomer PRs closed without merge: 23
Returning PRs still open without any response: 17
